# Semana 10, Aula 01 - Simulado de Exploração e Análise de Vendas
**Base real:** Online Retail II — varejo online do Reino Unido (2009–2011)

Percorraremos as RFs do miniprojeto de forma mais enxuta.

### Onde baixar os dados brutos
Vamos realizar o carregamento dessa base de dados, porém, primeiro precisamos baixa-la usando o URL abaixo:
- Kaggle: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci

In [1]:
# Passo 1 — Importações 
import pandas as pd
import numpy as np
import re, os, json
import matplotlib.pyplot as plt
import seaborn as sns

## RF01 — Carregar o dataset de vendas

In [2]:
def carregar_dados(caminho="../data/raw/online_retail_II.csv"):
    """Lê o CSV bruto da Online Retail II."""
    df = pd.read_csv(caminho)
    print(f"{len(df)} linhas carregadas.")
    return df

df_bruto = carregar_dados()
df_bruto.head()

1067371 linhas carregadas.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## RF02 — Inspecionar e descrever os dados

In [3]:
def inspecionar_dados(df):
    print("INSPECAO INICIAL")
    print(f"Shape: {df.shape}")
    print(f"Colunas: {list(df.columns)}")
    print(f"\nTipos:\n{df.dtypes}")
    print(f"\nNulos por coluna:\n{df.isnull().sum()}")

    return df.describe().round(2)

inspecionar_dados(df_bruto)

INSPECAO INICIAL
Shape: (1067371, 8)
Colunas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Tipos:
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

Nulos por coluna:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


,Quantity,Price,Customer ID
count,1067371.00,1067371.00,824364.00
mean,9.94,4.65,15324.64
std,172.71,123.55,1697.46
min,-80995.00,-53594.36,12346.00
25%,1.00,1.25,13975.00
50%,3.00,2.10,15255.00
75%,10.00,4.15,16797.00
max,80995.00,38970.00,18287.00


## RF03 — Limpar e tratar os dados (regex + datas + nulos)
A Online Retail II vem suja: descrições/cliente nulos, **devoluções** (quantidade < 0)
e **preços ≤ 0**. Tratamos tudo e convertemos a data (InvoiceDate) para `datetime`.

In [4]:
import re
import pandas as pd

def normalizar_texto(texto):
    """
    Limpa um valor de texto:
    - troca qualquer sequencia de espacos por um unico espaco
    - remove os espacos das pontas
    """
    if pd.isna(texto):
        return texto
    texto = str(texto)
    texto = re.sub(r"\s+", " ", texto)   # colapsa espacos repetidos
    return texto.strip()                 # remove espacos das pontas

def limpar_strings(df, colunas):
    """Aplica a normalizacao de texto nas colunas indicadas."""
    df = df.copy()
    for coluna in colunas:
        df[coluna] = df[coluna].apply(normalizar_texto)
    return df

def limpar_dados(df):
    """
    Limpa o DataFrame de vendas em etapas e devolve (df_limpo, relatorio).
    O relatorio guarda quantas linhas cada etapa removeu, para mostrar o impacto.
    """
    df = df.copy()
    relatorio = {}
    relatorio["registros_iniciais"] = len(df)

    # Etapa 1: normalizar textos (espacos extras em Description e Country) 
    df = limpar_strings(df, ["Description", "Country"])

    # Etapa 2: converter a coluna de data para o tipo datetime, e remover linhas sem data valida 
    # errors="coerce" transforma datas invalidas em vazio (NaT)
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    antes = len(df)
    df = df.dropna(subset=["InvoiceDate"])      # remove linhas sem data valida
    relatorio["datas_invalidas_removidas"] = antes - len(df)

    # Etapa 3: remover linhas sem cliente ou sem descricao
    # Sem Customer ID nao da para segmentar o cliente.
    antes = len(df)
    df = df.dropna(subset=["Customer ID", "Description"])
    relatorio["nulos_removidos"] = antes - len(df)

    # Etapa 4: remover devolucoes e precos invalidos
    # Quantity <= 0  -> devolucao / cancelamento
    # Price    <= 0  -> ajuste contabil ou erro (nao é uma venda real)
    antes = len(df)
    df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
    relatorio["devolucoes_e_precos_invalidos_removidos"] = antes - len(df)

    # Etapa 5: ajustar o tipo do Customer ID
    # Estava como float por causa dos nulos; agora que nao ha mais nulos, vira int.
    df["Customer ID"] = df["Customer ID"].astype(int)

    # Resumo final
    relatorio["registros_finais"] = len(df)
    relatorio["registros_removidos_total"] = relatorio["registros_iniciais"] - len(df)

    print("RELATORIO DE LIMPEZA")
    for etapa, valor in relatorio.items():
        print(f"  {etapa}: {valor}")

    return df, relatorio

df_v1, relatorio = limpar_dados(df_bruto)
df_v1.to_csv("../data/processed/v1_com_outliers/vendas_v1.csv", index=False)
print("\nv1 salva em ../data/processed/v1_com_outliers/")

RELATORIO DE LIMPEZA
  registros_iniciais: 1067371
  datas_invalidas_removidas: 0
  nulos_removidos: 243007
  devolucoes_e_precos_invalidos_removidos: 18815
  registros_finais: 805549
  registros_removidos_total: 261822

v1 salva em ../data/processed/v1_com_outliers/


## RF04 — Outliers (IQR): versões v1 e v2

In [5]:
def tratar_outliers(df, colunas, fator=1.5, metodo="remover"):
    """Trata outliers numericos via IQR. metodo='remover' ou 'limitar'."""
    df = df.copy()
    for col in colunas:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lim_inf, lim_sup = q1 - fator*iqr, q3 + fator*iqr
        n_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
        print(f"  {col}: {n_out} outliers (lim_inf={lim_inf:.2f}, lim_sup={lim_sup:.2f})")
        if metodo == "remover":
            df = df[(df[col] >= lim_inf) & (df[col] <= lim_sup)]
        else:
            df[col] = df[col].clip(lower=lim_inf, upper=lim_sup)
    return df

tmp = df_v1.copy(); tmp["receita_total"] = tmp["Quantity"]*tmp["Price"]
df_v2 = tratar_outliers(tmp, colunas=["Quantity", "receita_total"], metodo="remover")
df_v2 = df_v2.drop(columns=["receita_total"])
print(f"v1={len(df_v1)} | v2={len(df_v2)} | removidas={len(df_v1)-len(df_v2)}")
df_v2.to_csv("../data/processed/v2_outliers_tratado/vendas_v2.csv", index=False)
print("v2 salva.")

  Quantity: 51983 outliers (lim_inf=-13.00, lim_sup=27.00)
  receita_total: 40192 outliers (lim_inf=-15.93, lim_sup=37.88)
v1=805549 | v2=713374 | removidas=92175
v2 salva.
